# 08, Baseline comparison

Per `plan.md` §11 step 11. Render the §9 ablation table side-by-
side with bootstrapped 95% CIs from `results/ablations.csv`. The
rows include both the CNN ablations (written by
`scripts/train_logo.py`) and the classical baselines (written by
`scripts/run_baseline.py`).

Run after both scripts have populated `ablations.csv`.

In [ ]:
%matplotlib inline
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hishells.eval import aggregate_folds, bootstrap_ci

In [ ]:
REPO = Path('..').resolve()
csv_path = REPO / 'results' / 'ablations.csv'
df = pd.read_csv(csv_path)
df.head()

## Per-ablation summary

Mean ± 95% CI across the 19 LOGO folds for the headline
operating-point metrics.

In [ ]:
rows = []
for name, sub in df.groupby('name'):
    agg = aggregate_folds(sub.to_dict('records'))
    rows.append({
        'name': name,
        'n_folds': len(sub),
        'recall_mean': agg['mean_recall_at_op'],
        'recall_ci': agg['ci_recall_at_op'],
        'fp_mean': agg['mean_fp_per_galaxy_at_op'],
        'fp_ci': agg['ci_fp_per_galaxy_at_op'],
        'AUC_PR': agg['mean_AUC_PR'],
        'AUC_ROC': agg['mean_AUC_ROC'],
        'ECE': agg['mean_ECE'],
        'pass_recall': agg['passes_recall_target'],
        'pass_fp': agg['passes_fp_target'],
    })
summary = pd.DataFrame(rows).sort_values('recall_mean', ascending=False).reset_index(drop=True)
summary

## Recall vs FP/galaxy

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, max(3, 0.4 * len(summary) + 2)))
y = np.arange(len(summary))
axes[0].errorbar(
    summary['recall_mean'], y,
    xerr=[
        summary['recall_mean'] - [c[0] for c in summary['recall_ci']],
        [c[1] for c in summary['recall_ci']] - summary['recall_mean'],
    ],
    fmt='o', capsize=3,
)
axes[0].axvline(0.70, color='red', linestyle=':', label='target ≥ 0.70')
axes[0].set_yticks(y); axes[0].set_yticklabels(summary['name'])
axes[0].invert_yaxis(); axes[0].set_xlim(0, 1)
axes[0].set_xlabel('recall @ op (mean ± 95% CI)')
axes[0].legend(fontsize='small')

axes[1].errorbar(
    summary['fp_mean'], y,
    xerr=[
        summary['fp_mean'] - [c[0] for c in summary['fp_ci']],
        [c[1] for c in summary['fp_ci']] - summary['fp_mean'],
    ],
    fmt='o', capsize=3, color='C1',
)
axes[1].axvline(5.0, color='red', linestyle=':', label='target ≤ 5')
axes[1].set_yticks(y); axes[1].set_yticklabels(['' for _ in summary['name']])
axes[1].invert_yaxis()
axes[1].set_xlabel('FP / galaxy @ op (mean ± 95% CI)')
axes[1].legend(fontsize='small')
fig.suptitle('LOGO sweep, baselines vs CNN ablations')
fig.tight_layout()

## Per-fold detail (one ablation at a time)

In [ ]:
PICK = summary['name'].iloc[0]
sub = df[df['name'] == PICK].sort_values('fold')
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(sub['fold'], sub['recall_at_op'])
axes[0].axhline(0.70, color='red', linestyle=':', label='target ≥ 0.70')
axes[0].set_title(f'{PICK}, recall_at_op'); axes[0].set_xticklabels(sub['fold'], rotation=45, ha='right')
axes[0].legend(fontsize='small')
axes[1].bar(sub['fold'], sub['fp_per_galaxy_at_op'], color='C1')
axes[1].axhline(5.0, color='red', linestyle=':', label='target ≤ 5')
axes[1].set_title(f'{PICK}, fp/galaxy_at_op'); axes[1].set_xticklabels(sub['fold'], rotation=45, ha='right')
axes[1].legend(fontsize='small')
fig.tight_layout()

## Calibration & threshold-free AUCs

In [ ]:
metrics = ['AUC_PR', 'AUC_ROC', 'ECE']
summary[['name'] + metrics].set_index('name').plot.bar(
    figsize=(10, 4), subplots=True, layout=(1, 3), legend=False, sharex=True,
)
plt.tight_layout()